In [ ]:
#! pip install langchain==0.3.26
#! pip install flask==3.1.1 
#! pip install sentence-transformers==4.1.0
#! pip install pypdf==5.6.1 
#! pip install python-dotenv==1.1.0
#! pip install langchain-pinecone==0.2.8 
#! pip install langchain-openai==0.3.24
#! pip install langchain-community==0.3.26
# pip install dotenv
#! pip install langchain-openai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
%pwd

'/Users/hassanfarid/Documents/Pakistan-Constitution-Chatbot/research'

In [3]:
import os
os.chdir('../')

In [4]:
%pwd

'/Users/hassanfarid/Documents/Pakistan-Constitution-Chatbot'

In [9]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [11]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls = PyPDFLoader
    )
    documents = loader.load()
    return documents

data_path = 'dataset/'

extracted_data = load_pdf_files(data_path)

In [14]:
extracted_data[0].page_content

'THE \nCONSTITUTION \nOF THE \nISLAMIC REPUBLIC \nOF \nPAKISTAN \n \n \n \n \n \n \n[As modified upto the  31st May, 2018] \n \n \n \n \n \n \nNATIONAL ASSEMBLY OF PAKISTAN'

In [15]:
print("Length of data(pages):",len(extracted_data)) 

Length of data(pages): 225


In [16]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only "source" in the metadata and the original page_content.
    """
    
    minimal_docs: List[Document] = []
    
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source": src}
            )
        )
    return minimal_docs

minimal_docs = filter_to_minimal_docs(extracted_data)

In [19]:
minimal_docs[0]

Document(metadata={'source': 'dataset/THE _CONSTITUTION_OF_THE_ISLAMIC_REPUBLIC_OF_PAKISTAN.pdf'}, page_content='THE \nCONSTITUTION \nOF THE \nISLAMIC REPUBLIC \nOF \nPAKISTAN \n \n \n \n \n \n \n[As modified upto the  31st May, 2018] \n \n \n \n \n \n \nNATIONAL ASSEMBLY OF PAKISTAN')

In [85]:
def text_split(docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20
    )
    text_chunks = text_splitter.split_documents(docs)
    return text_chunks

text_chunks = text_split(minimal_docs)

In [86]:
text_chunks = text_split(minimal_docs)
print(f'Number of Chunks: {len(text_chunks)}')

Number of Chunks: 1085


In [87]:
from langchain.embeddings import HuggingFaceEmbeddings


def download_embeddings():
    """
    Download and return huggingface embedding model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )
    return embeddings
    
embedding = download_embeddings()

In [88]:
#from langchain_openai import OpenAIEmbeddings

#embedding = OpenAIEmbeddings(model = 'text-embedding-3-small')
#embedding

In [89]:
print(embedding)
vectors = embedding.embed_query("Hello, World")

print(f"Vectors: {vectors}")
print(f"Length of Vectors: {len(vectors)}")

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
) model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False
Vectors: [-0.056094396859407425, 0.03554259240627289, 0.004592870362102985, 0.02386585623025894, -0.04944375529885292, -0.15514132380485535, 0.06592568755149841, 0.022491320967674255, -0.021727290004491806, 0.014119270257651806, 0.05505150556564331, 0.024055687710642815, 0.005019017029553652, -0.006477480288594961, -0.03411555662751198, -0.05552103742957115, -0.0067526232451200485, -0.02301395684480667, -0.

In [99]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ['PINECONE_API_KEY'] = PINECONE_API_KEY
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [100]:
from pinecone import Pinecone
pinecine_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecine_api_key)

In [101]:
from pinecone import ServerlessSpec

index_name = "constitution-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension = 384,
        metric = 'cosine',
        spec = ServerlessSpec(cloud='aws', region= 'us-east-1')
    )
    
index = pc.Index(index_name)

In [102]:
index

In [103]:
from langchain_pinecone import PineconeVectorStore

doc_search = PineconeVectorStore.from_documents(
    documents = text_chunks,
    embedding = embedding,
    index_name = index_name,

)

In [104]:
retriever = doc_search.as_retriever(search_type='similarity', search_kwargs={"k":3})

In [105]:
retriev_docs = retriever.invoke("What fundamental rights are guaranteed to citizens under the Constitution of Pakistan?")

In [106]:
type(retriev_docs)
retriev_docs[0].page_content

'CONSTITUTION OF PAKISTAN  \n 2 \nWherein the integrity of the territories of the Federation, its \nindependence and all its rights, including its sovereign rights on land, sea \nand air, shall be safeguarded; \n \nSo that the people of Pakistan m ay prosper and attain their rightful \nand honoured place amongst the nations of the World and make their full \ncontribution towards international peace and progress and happiness of \nhumanity; \n \nNow, therefore, we, the people of Pakistan;'

In [107]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    model = 'gpt-4o'
)


In [108]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [123]:
from langchain.prompts import ChatPromptTemplate

system_prompt = (
    "You are a law-focused assistant specializing in the Constitution of the Islamic Republic of Pakistan. "
    "Your task is to answer questions in a way that is detailed, clear, and easy to understand, even for people "
    "who have no background in law. Always base your responses strictly on the retrieved context provided.\n\n"
    "When answering a question:\n"
    "- Explain the concept in simple, beginner-friendly language without losing legal accuracy.\n"
    "- Provide a detailed explanation, breaking down complex legal terms into plain words.\n"
    "- Identify and cite the most relevant constitutional Article(s) and Section(s).\n"
    "- Where helpful, give short examples or analogies to make the explanation easier.\n"
    "- If the context does not provide enough information, reply only with: 'I don't know.'\n\n"
    "Format your response as:\n"
    "Answer:\n<Your detailed and beginner-friendly explanation here>\n\n"
    "Relevant Article(s): <List the article numbers/sections if available, else 'Not found'>\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)


In [124]:
question_answere_chain = create_stuff_documents_chain(chat_model,prompt)
rag_chain = create_retrieval_chain(retriever, question_answere_chain)

In [125]:
response = rag_chain.invoke({"input": 'Police arrest me on the charges of fake blasphemy case, how can I secure myself?'})
response

{'input': 'Police arrest me on the charges of fake blasphemy case, how can I secure myself?',
 'context': [Document(id='fae4427d-a0ff-448f-8dfb-af044c069bfb', metadata={'source': 'dataset/THE _CONSTITUTION_OF_THE_ISLAMIC_REPUBLIC_OF_PAKISTAN.pdf'}, page_content='nine hundred and fifty-six, an offence. \n \n13. Protection against double punishment and self incrimination \n \n13. No person— \n \n(a) shall be prosecuted or punished for the same \noffence more than once; or \n \n(b) shall, when accused of an offence, be compelled to \nbe a witness against himself. \n \n14. Inviolability of dignity of man, etc. \n \n14. (1) The dignity of man and, subject to law, the privacy of \nhome, shall be inviolable. \n \n(2) No person shall be subjected to torture for the purpose of'),
  Document(id='fe9a7037-b77d-46a5-af9b-705399e34b15', metadata={'source': 'dataset/THE _CONSTITUTION_OF_THE_ISLAMIC_REPUBLIC_OF_PAKISTAN.pdf'}, page_content='(2) Every person who is arrested and detained in custody sha

In [126]:
print(response['answer'])

Answer:
If you find yourself wrongly arrested on false charges, such as a blasphemy case, there are certain rights and protections under the Constitution of Pakistan that you should be aware of to secure yourself:

1. **Right against Double Jeopardy and Self-incrimination**: According to Article 13 of the Constitution, no person can be prosecuted or punished for the same offense more than once (double jeopardy). Also, you cannot be forced to testify against yourself, ensuring that your rights are protected during legal proceedings.

2. **Inviolability of Dignity and Protection from Torture**: Article 14(1) of the Constitution ensures that the dignity of every person is protected, and Article 14(2) states that no person should be subjected to torture. This means that during your arrest and detention, your dignity must be respected, and you should not face any form of torture.

3. **Right to be Brought before a Magistrate**: The law requires that anyone arrested must be brought before a 